In [1]:
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
import os
import time 
import pandas as pd 
import dotenv
dotenv.load_dotenv()
from transformers import AutoModel, AutoTokenizer
import torch
import unicodedata
import re

c:\Users\Andrew\AppData\Local\Programs\Python\Python311\Lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
runpod_token = os.getenv("RUNPOD_TOKEN")
endpoint_id = os.getenv("RUNPOD_EMBEDDING_ID")
model_name = os.getenv("MODEL_NAME")
base_url = f"https://api.runpod.ai/v2/{endpoint_id}/openai/v1"
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX_NAME")

In [3]:
pc = Pinecone(api_key=pinecone_api_key)
client = OpenAI(api_key=runpod_token, base_url=base_url)

In [ ]:
print(model_name)
hf_token = os.getenv("HF_TOKEN")

meta-llama/Meta-Llama-3-8B-Instruct


# Try Embeddings

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token = hf_token)
model = AutoModel.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16, use_auth_token = hf_token)

c:\Users\Andrew\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\auto\tokenization_auto.py:786: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
c:\Users\Andrew\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\auto\auto_factory.py:469: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.31s/it]


In [7]:
output = client.embeddings.create(input = ['hello world'], model = model_name)
embeddings = output.data[0].embedding
print(embeddings)

[0.016397669911384583, -0.02263857237994671, 0.007892907597124577, -0.0744013637304306, 0.004160602577030659, 0.0034416751004755497, -0.031571630388498306, 0.045032404363155365, 0.04356395825743675, -0.007739944849163294, -0.024596504867076874, -0.03377430513501167, 0.014317368157207966, 0.04576662927865982, 0.008321205154061317, -0.016030557453632355, 0.0076787592843174934, -0.019946418702602386, -0.11355997622013092, -0.017499005421996117, 0.12628652155399323, 0.029246589168906212, 0.026064952835440636, -0.03426378592848778, -0.041361283510923386, 0.007005720864981413, 0.010340320877730846, 0.023372797295451164, 0.004007639363408089, -0.12726549804210663, -0.015663444995880127, -0.02043590135872364, 0.048703525215387344, 0.012114696204662323, 0.06803809106349945, 0.007739944849163294, -0.01798848807811737, 0.041606027632951736, -0.008994244039058685, 0.023862279951572418, 0.0108298035338521, -0.028267623856663704, 0.00807646382600069, -0.015479889698326588, 0.03047029674053192, -0.06

# Wrangle Dataset

In [8]:
df = pd.read_json('products/products.jsonl', lines=True)
df_syrup = pd.read_json('products/syrups.jsonl', lines = True)

In [ ]:
df.head(5)

In [9]:
df['text'] = df['name'] + ' : ' + df['description'] +\
    ' -- Category: ' + df['category'].astype(str) +\
    ' -- Ingredients: ' + df['ingredients'].astype(str) +\
    ' -- Price: ' + df['price'].astype(str) +\
    ' -- Rating: ' + df['rating'].astype(str) +\
    ' -- Calories: ' + df['calories'].astype(str) +\
    ' -- Sizes: ' + df['sizes'].astype(str) +\
    ' -- Syrups: ' + df['Syrups'].astype(str)

In [10]:
df_syrup['text'] = df_syrup['name'] + ' : ' + df_syrup['description'] +\
    ' -- Category: ' + df_syrup['category'].astype(str) +\
    ' -- Ingredients: ' + df_syrup['ingredients'].astype(str) +\
    ' -- Calories: ' + df_syrup['calories'].astype(str) +\
    ' -- Sizes: ' + df_syrup['sizes'].astype(str)

In [11]:
df_syrup['text'].head()

0    Chocolate syrup : Our rich chocolate syrup is ...
1    Hazelnut syrup : Add a nutty flavor to your dr...
2    Carmel syrup : Sweet and creamy, our caramel s...
3    Sugar Free Vanilla syrup : Enjoy the sweet fla...
Name: text, dtype: object

In [12]:
texts = df['text'].tolist()
texts.extend(df_syrup['text'].tolist())
texts

["Cappuccino : A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations. -- Category: Coffee -- Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam'] -- Price: {'Small': 3.75, 'Medium': 4.5, 'Large': 5.25} -- Rating: 4.7 -- Calories: {'Small': 80, 'Medium': 110, 'Large': 150} -- Sizes: ['Small', 'Medium', 'Large'] -- Syrups: ['Chocolate syrup', 'Hazelnut syrup', 'Carmel syrup', 'Sugar Free Vanilla syrup']",
 "Jumbo Savory Scone : Deliciously flaky and buttery, this jumbo savory scone is filled with herbs and cheese, creating a mouthwatering experience. Perfect for a hearty snack or a light lunch, it pairs beautifully with your favorite coffee or tea. -- Category: Bakery -- Ingredients: ['Flour', 'Butter', 'Cheese', 'Herbs', 'Baking Powder', 'Salt'] -- Price: 3.25 -- Rating: 4.3 -- Ca

In [13]:
with open('products/Andrew_Cafe_about_us.txt') as file:
    Merry_way_about_section = file.read()

Merry_way_about_section = "Coffee shop Andrews Cafe about section: " + Merry_way_about_section
texts.append(Merry_way_about_section)


In [14]:
with open('products/menu_items_text.txt') as file:
    menu_items_text = file.read()

menu_items_text = 'Menu Items: ' + menu_items_text
texts.append(menu_items_text)

# Generate Embeddings

In [15]:
output = client.embeddings.create(input = texts, model = model_name)

In [16]:
embeddings = output.data

# Push data to Pinecone

In [ ]:
pc.create_index(
    name=pinecone_index_name,
    dimension=len(embeddings[0].embedding),
    metric="cosine",
    spec=ServerlessSpec(cloud = 'aws', region = 'us-east-1')
)

In [ ]:
def sanitize_id(text):
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")

    text = text.lower()

    text = re.sub(r'[^a-z0-9]+', '_', text)

    text = text.strip('_')

    return text or "id"

In [17]:
#Wait for the index to be ready
while not pc.describe_index(pinecone_index_name).status['ready']:
    time.sleep(5)

index = pc.Index(pinecone_index_name)

vectors = []
for text, e in zip(texts, embeddings):
    entry_id = text.split(':')[0]
    vectors.append({
        'id': entry_id,
        'values': e.embedding,
        'metadata': {'text': text}
    })

index.upsert(vectors, namespace = 'ns1')

{'upserted_count': 21}

# Get Closest Documents

In [ ]:
output = client.embeddings.create(input = ['Is Cappuccino lactose-free?'], model = model_name)
embedding = output.data[0].embedding

In [ ]:
index = pc.Index(pinecone_index_name)
results = index.query(namespace = 'ns1', vector = embedding, top_k = 3, include_values = False, include_metadata = True)
print(results)